# Machine Learning Group Assignment – Hotel Booking Cancellation

**Reference structure:** This notebook follows the same ML concepts and workflow used in the reference Financial Loan assignment: dataset overview, descriptive statistics, frequency distributions, missing-value analysis, outlier detection, correlation analysis, relationship analysis, feature engineering, one-hot encoding, standard scaling, SMOTE, feature selection/multicollinearity assessment, Logistic Regression, Decision Tree, Random Forest, confusion matrix, Accuracy, Precision, Recall, F1 Score, ROC-AUC, feature interpretation and model comparison.

**Dataset:** `hotel_bookings.csv`
**Target:** `is_canceled` (0 = Not Canceled, 1 = Canceled)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# The CSV is read from Google Drive, not uploaded directly to Colab.
# The code searches MyDrive for hotel_bookings.csv so the folder name does not have to be changed.
matches = glob.glob('/content/drive/MyDrive/**/hotel_bookings.csv', recursive=True)
if len(matches) == 0:
    raise FileNotFoundError(
        'hotel_bookings.csv was not found in Google Drive. Place the file in My Drive and run this cell again.'
    )

DATA_PATH = matches[0]
print('Data file used:', DATA_PATH)
df = pd.read_csv(DATA_PATH)
display(df.head())

## 1. Dataset Analysis

In [ ]:
print(f'Dataset Shape: {df.shape}')
print('\nColumn Names:')
print(df.columns.tolist())

In [ ]:
# Variable summary, similar to the reference assignment
summary_df = pd.DataFrame({
    'Variable Name': df.columns,
    'Variable Type': df.dtypes.values,
    'Missing Values': df.isnull().sum().values,
    'Missing Value %': (df.isnull().sum() / len(df) * 100).round(2).values
})

target_variable = 'is_canceled'
numerical_vars = df.select_dtypes(include=np.number).columns.tolist()
categorical_vars = df.select_dtypes(include=['object', 'category']).columns.tolist()

summary_df['Role'] = 'Other'
summary_df.loc[summary_df['Variable Name'] == target_variable, 'Role'] = 'Target'
summary_df.loc[summary_df['Variable Name'].isin(numerical_vars) & (summary_df['Variable Name'] != target_variable), 'Role'] = 'Numerical Feature'
summary_df.loc[summary_df['Variable Name'].isin(categorical_vars), 'Role'] = 'Categorical Feature'

display(summary_df)
print('\nNumerical variables:', numerical_vars)
print('Categorical variables:', categorical_vars)

## 2. Target Variable

In [ ]:
print('Target variable:', target_variable)
print('\nTarget frequency:')
display(df[target_variable].value_counts().to_frame('Frequency'))
print('\nTarget percentage distribution:')
display((df[target_variable].value_counts(normalize=True) * 100).round(2).to_frame('Percentage'))

plt.figure(figsize=(7,5))
sns.countplot(data=df, x=target_variable, hue=target_variable, legend=False)
plt.title('Distribution of Hotel Booking Cancellation')
plt.xlabel('is_canceled')
plt.ylabel('Count')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()

## 3. Descriptive Statistics for Numerical Variables

In [ ]:
numerical_vars = df.select_dtypes(include=np.number).columns.tolist()
descriptive_stats = df[numerical_vars].describe().T
descriptive_stats['median'] = df[numerical_vars].median()
descriptive_stats = descriptive_stats[['count','mean','median','std','min','25%','50%','75%','max']]
display(descriptive_stats)

## 4. Visualizing Numerical Variables

In [ ]:
for var in numerical_vars:
    print(f'### Variable: {var}')
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle(f'Distribution of {var}', fontsize=16)
    sns.histplot(df[var].dropna(), kde=True, ax=axes[0], bins=30)
    axes[0].set_title(f'Histogram of {var}')
    axes[0].set_xlabel(var)
    axes[0].set_ylabel('Frequency')
    axes[0].grid(True, linestyle='--', alpha=0.6)
    sns.boxplot(y=df[var], ax=axes[1])
    axes[1].set_title(f'Boxplot of {var}')
    axes[1].set_ylabel(var)
    axes[1].grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

## 5. Frequency Distribution Analysis for Categorical Variables

In [ ]:
# reservation_status and reservation_status_date are shown separately only as data-quality/context fields
# because they are post-outcome variables and should not be used as predictors of is_canceled.
for var in categorical_vars:
    print(f'### Variable: {var}')
    freq_table = df[var].value_counts(dropna=False)
    print('Frequency Table:')
    display(freq_table.to_frame('Frequency'))
    print('Percentage Distribution:')
    display((df[var].value_counts(dropna=False, normalize=True) * 100).round(2).to_frame('Percentage'))
    plt.figure(figsize=(12, 6))
    order = df[var].value_counts(dropna=False).index
    sns.countplot(data=df, y=var, order=order, hue=var, legend=False)
    plt.title(f'Bar Chart of {var}')
    plt.xlabel('Count')
    plt.ylabel(var)
    plt.grid(axis='x', linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

## 6. Detailed Missing Value Analysis

In [ ]:
missing_data = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing Percentage': (df.isnull().sum() / len(df)) * 100
})
missing_data = missing_data[missing_data['Missing Count'] > 0].sort_values(by='Missing Percentage', ascending=False)
print('Missing Value Summary:')
if missing_data.empty:
    print('No missing values found in the dataset.')
else:
    display(missing_data)

In [ ]:
plt.figure(figsize=(14, 6))
sns.heatmap(df.isnull(), cbar=False)
plt.title('Heatmap of Missing Values')
plt.xlabel('Columns')
plt.ylabel('Rows')
plt.show()

## 7. Outlier Detection Using IQR

In [ ]:
for var in numerical_vars:
    if var == target_variable:
        continue
    print(f'### Variable: {var}')
    series = df[var].dropna()
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[var] < lower_bound) | (df[var] > upper_bound)]
    outlier_count = outliers.shape[0]
    outlier_percentage = (outlier_count / len(df)) * 100
    print(f'IQR: {IQR:.2f}')
    print(f'Lower Bound: {lower_bound:.2f}')
    print(f'Upper Bound: {upper_bound:.2f}')
    print(f'Number of Outliers: {outlier_count}')
    print(f'Percentage of Outliers: {outlier_percentage:.2f}%\n')

## 8. Correlation Analysis

In [ ]:
corr_vars = [v for v in numerical_vars if v != target_variable]
correlation_matrix = df[corr_vars].corr()
print('Correlation Matrix:')
display(correlation_matrix)

plt.figure(figsize=(14, 11))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=.5)
plt.title('Correlation Heatmap of Numerical Variables')
plt.show()

## 9. Relationship Analysis with `is_canceled`

In [ ]:
for var in corr_vars:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=target_variable, y=var, data=df, hue=target_variable, legend=False)
    plt.title(f'{var} by is_canceled')
    plt.xlabel('is_canceled')
    plt.ylabel(var)
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

In [ ]:
# Categorical relationship analysis.
# Post-outcome fields are excluded because they directly describe the cancellation outcome.
categorical_vars_for_analysis = [
    v for v in categorical_vars
    if v not in ['reservation_status', 'reservation_status_date']
]

for var in categorical_vars_for_analysis:
    plt.figure(figsize=(12, 6))
    sns.countplot(data=df, y=var, hue=target_variable, order=df[var].value_counts().index)
    plt.title(f'Distribution of {var} by is_canceled')
    plt.xlabel('Count')
    plt.ylabel(var)
    plt.grid(axis='x', linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

## 10. Data Preprocessing and Feature Engineering

### Feature Engineering: Arrival Date

The hotel dataset already stores arrival year, month and day separately. To follow the reference notebook's date-feature approach, these fields are combined into an arrival date and then represented again as numerical date components. The original text month is not retained as a modelling field.

In [ ]:
month_map = {
    'January':1, 'February':2, 'March':3, 'April':4, 'May':5, 'June':6,
    'July':7, 'August':8, 'September':9, 'October':10, 'November':11, 'December':12
}

df['Arrival_Date'] = pd.to_datetime({
    'year': df['arrival_date_year'],
    'month': df['arrival_date_month'].map(month_map),
    'day': df['arrival_date_day_of_month']
})

df['Arrival_Year'] = df['Arrival_Date'].dt.year
df['Arrival_Month'] = df['Arrival_Date'].dt.month
df['Arrival_Day'] = df['Arrival_Date'].dt.day

df = df.drop(columns=['Arrival_Date', 'arrival_date_year', 'arrival_date_month', 'arrival_date_day_of_month'])

print('New date features created: Arrival_Year, Arrival_Month, Arrival_Day')
display(df[['Arrival_Year','Arrival_Month','Arrival_Day']].head())

### Encoding Categorical Variables

One-hot encoding is applied with `drop_first=True`, following the reference notebook's approach.

In [ ]:
# Post-outcome variables are not predictors of cancellation and are excluded from modelling.
# Agent and company are administrative identifiers, analogous to the ID field in the reference assignment.
# They are also excluded from the model.

leakage_or_identifier_cols = ['reservation_status', 'reservation_status_date', 'agent', 'company']

# Keep a copy for modelling before encoding
df_model = df.copy()
df_model = df_model.drop(columns=[c for c in leakage_or_identifier_cols if c in df_model.columns])

# Separate target
y = df_model[target_variable].astype(int)
X_raw = df_model.drop(columns=[target_variable])

# Identify categorical columns
categorical_cols = X_raw.select_dtypes(include=['object', 'category']).columns.tolist()
print('Categorical columns to be one-hot encoded:')
print(categorical_cols)

# Fill missing categorical values using the most frequent category, as a missing-value safeguard.
for col in categorical_cols:
    if X_raw[col].isnull().any():
        X_raw[col] = X_raw[col].fillna(X_raw[col].mode(dropna=True)[0])

# One-hot encoding
df_encoded = pd.get_dummies(X_raw, columns=categorical_cols, drop_first=True, dtype=int)
print('Shape after one-hot encoding:', df_encoded.shape)
display(df_encoded.head())

### Missing-Value Safeguard for Numerical Variables

In [ ]:
# Median imputation for numerical variables, following the reference assignment's preprocessing safeguard.
numerical_feature_cols = df_encoded.select_dtypes(include=np.number).columns.tolist()
for col in numerical_feature_cols:
    if df_encoded[col].isnull().any():
        df_encoded[col] = df_encoded[col].fillna(df_encoded[col].median())

print('Remaining missing values after preprocessing:', int(df_encoded.isnull().sum().sum()))

### Scaling Numerical Variables

In [ ]:
from sklearn.preprocessing import StandardScaler

X = df_encoded.copy()

# Scale numerical predictors, as in the reference notebook.
numerical_cols_for_scaling = X.select_dtypes(include=np.number).columns.tolist()

scaler = StandardScaler()
X[numerical_cols_for_scaling] = scaler.fit_transform(X[numerical_cols_for_scaling])

print('Numerical columns scaled:', len(numerical_cols_for_scaling))
display(X[numerical_cols_for_scaling].describe().T.head(15))

## 11. Addressing Class Imbalance with SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE
from collections import Counter

print('Original class distribution:', Counter(y))

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

print('Resampled class distribution:', Counter(y_resampled))
print('Shape of X_resampled:', X_resampled.shape)
print('Shape of y_resampled:', y_resampled.shape)

### Encoding Strategy Review

In [ ]:
print('Target values:', sorted(y_resampled.unique()))
print('Target class distribution after SMOTE:')
display(y_resampled.value_counts().sort_index().to_frame('Count'))

print('Feature matrix shape before feature selection:', X_resampled.shape)

## 12. Feature Selection and Multicollinearity Assessment

Following the reference assignment, administrative identifiers and variables that can leak the outcome or represent post-outcome information are excluded. The remaining variables are retained for modelling.

In [ ]:
# Remove administrative/post-outcome fields if present.
# reservation_status and reservation_status_date were already removed before encoding.
# agent and company were also removed because they are administrative identifiers.

remove_cols = [c for c in ['ID', 'agent', 'company', 'reservation_status', 'reservation_status_date'] if c in X_resampled.columns]
if remove_cols:
    X_resampled = X_resampled.drop(columns=remove_cols)

print('Columns removed during feature selection:', remove_cols if remove_cols else 'None remaining to remove')
print('Updated feature matrix shape:', X_resampled.shape)

## 13. Logistic Regression

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc, classification_report

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.30, random_state=42, stratify=y_resampled
)

print('X_train:', X_train.shape, 'X_test:', X_test.shape)
print('Training class distribution:', y_train.value_counts().to_dict())
print('Testing class distribution:', y_test.value_counts().to_dict())

log_reg_model = LogisticRegression(solver='lbfgs', random_state=42, max_iter=1000)
log_reg_model.fit(X_train, y_train)

y_pred = log_reg_model.predict(X_test)
y_prob = log_reg_model.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_pred)
print('Confusion Matrix:')
display(pd.DataFrame(cm, index=['Actual 0','Actual 1'], columns=['Predicted 0','Predicted 1']))

print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Not Canceled','Canceled']))

fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc_lr = auc(fpr, tpr)
print(f'ROC-AUC Score: {roc_auc_lr:.4f}')

plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, label=f'Logistic Regression (AUC = {roc_auc_lr:.2f})')
plt.plot([0,1],[0,1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Logistic Regression')
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

coefficients = log_reg_model.coef_[0]
feature_importance_lr = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': coefficients,
    'Absolute_Coefficient': np.abs(coefficients)
}).sort_values('Absolute_Coefficient', ascending=False)

print('Top 10 influential variables by absolute coefficient:')
display(feature_importance_lr.head(10))

## 14. Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

dt_classifier = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_classifier.fit(X_train, y_train)

y_pred_dt = dt_classifier.predict(X_test)
y_prob_dt = dt_classifier.predict_proba(X_test)[:, 1]

plt.figure(figsize=(25, 15))
plot_tree(
    dt_classifier,
    feature_names=X_train.columns.tolist(),
    class_names=['Not Canceled','Canceled'],
    filled=True, rounded=True, fontsize=9, proportion=True
)
plt.title('Decision Tree Classifier (Max Depth = 5)')
plt.show()

feature_importance_dt = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': dt_classifier.feature_importances_
}).sort_values('Importance', ascending=False)
print('Top 10 Decision Tree features:')
display(feature_importance_dt.head(10))

cm_dt = confusion_matrix(y_test, y_pred_dt)
print('Confusion Matrix:')
display(pd.DataFrame(cm_dt, index=['Actual 0','Actual 1'], columns=['Predicted 0','Predicted 1']))
print('Classification Report:')
print(classification_report(y_test, y_pred_dt, target_names=['Not Canceled','Canceled']))

fpr_dt, tpr_dt, thresholds_dt = roc_curve(y_test, y_prob_dt)
roc_auc_dt = auc(fpr_dt, tpr_dt)
print(f'ROC-AUC Score: {roc_auc_dt:.4f}')

plt.figure(figsize=(8,6))
plt.plot(fpr_dt, tpr_dt, label=f'Decision Tree (AUC = {roc_auc_dt:.2f})')
plt.plot([0,1],[0,1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Decision Tree')
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

## 15. Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_classifier = RandomForestClassifier(
    n_estimators=100, max_depth=10, random_state=42, n_jobs=-1
)
rf_classifier.fit(X_train, y_train)

y_pred_rf = rf_classifier.predict(X_test)
y_prob_rf = rf_classifier.predict_proba(X_test)[:, 1]

cm_rf = confusion_matrix(y_test, y_pred_rf)
print('Confusion Matrix:')
display(pd.DataFrame(cm_rf, index=['Actual 0','Actual 1'], columns=['Predicted 0','Predicted 1']))
print('Classification Report:')
print(classification_report(y_test, y_pred_rf, target_names=['Not Canceled','Canceled']))

fpr_rf, tpr_rf, thresholds_rf = roc_curve(y_test, y_prob_rf)
roc_auc_rf = auc(fpr_rf, tpr_rf)
print(f'ROC-AUC Score: {roc_auc_rf:.4f}')

plt.figure(figsize=(8,6))
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {roc_auc_rf:.2f})')
plt.plot([0,1],[0,1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Random Forest')
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

feature_importance_rf = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf_classifier.feature_importances_
}).sort_values('Importance', ascending=False)
print('Top 10 Random Forest features:')
display(feature_importance_rf.head(10))

## 16. Model Comparison

In [ ]:
metrics = {
    'Logistic Regression': {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision (Canceled)': precision_score(y_test, y_pred, zero_division=0),
        'Recall (Canceled)': recall_score(y_test, y_pred, zero_division=0),
        'F1 Score (Canceled)': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_lr
    },
    'Decision Tree': {
        'Accuracy': accuracy_score(y_test, y_pred_dt),
        'Precision (Canceled)': precision_score(y_test, y_pred_dt, zero_division=0),
        'Recall (Canceled)': recall_score(y_test, y_pred_dt, zero_division=0),
        'F1 Score (Canceled)': f1_score(y_test, y_pred_dt, zero_division=0),
        'ROC-AUC': roc_auc_dt
    },
    'Random Forest': {
        'Accuracy': accuracy_score(y_test, y_pred_rf),
        'Precision (Canceled)': precision_score(y_test, y_pred_rf, zero_division=0),
        'Recall (Canceled)': recall_score(y_test, y_pred_rf, zero_division=0),
        'F1 Score (Canceled)': f1_score(y_test, y_pred_rf, zero_division=0),
        'ROC-AUC': roc_auc_rf
    }
}

comparison_df = pd.DataFrame(metrics).T
print('Model Comparison Table:')
display(comparison_df.round(4))

comparison_df[['Accuracy','Precision (Canceled)','Recall (Canceled)','F1 Score (Canceled)']].plot(kind='bar', figsize=(12,6))
plt.title('Model Performance Comparison')
plt.ylabel('Score')
plt.ylim(0,1)
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()

## 17. Interpretation of Model Results

Use the generated model comparison table, confusion matrices, ROC-AUC values, Logistic Regression coefficients and tree-based feature-importance tables to interpret the hotel-booking cancellation problem. The positive class is **Canceled (1)**.

The notebook does not hard-code model results; all reported metrics are generated from the student's dataset when the notebook is run.

## 18. Conclusion

This study applies the same core supervised-classification workflow used in the reference Financial Loan assignment to the hotel booking dataset. The analysis covers descriptive statistics, categorical frequency distributions, data-quality checks, outlier detection, correlation analysis, relationship analysis, feature engineering, one-hot encoding, scaling, SMOTE, feature selection, Logistic Regression, Decision Tree, Random Forest and standard classification evaluation measures.

The final model comparison should be based on the actual results produced after running the notebook in Google Colab.